<a href="https://colab.research.google.com/github/yanchenliu-cxk/ESM/blob/main/ESM_2_15B.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [2]:
import torch
from transformers import AutoTokenizer, EsmModel
from Bio import SeqIO
import numpy as np

# 1. 确认 A100 80GB 已就绪
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"正在使用的计算设备: {torch.cuda.get_device_name(0)}")
print(f"显存总量: {torch.cuda.get_device_properties(0).total_memory / 1e9:.2f} GB")

# 2. 加载 150 亿参数的 ESM-2 旗舰模型
model_name = "facebook/esm2_t48_15B_UR50D"
tokenizer = AutoTokenizer.from_pretrained(model_name)

# 关键优化：使用半精度(float16)并利用 accelerate 自动管理 80GB 显存
model = EsmModel.from_pretrained(
    model_name,
    torch_dtype=torch.float16,
    device_map="auto"
)
model.eval()

# 3. 读取你的 5252 条蛋白质序列
fasta_file = "/content/Galaxy36-[clean Prodigal protein].fasta"
protein_ids = []
protein_sequences = []

for record in SeqIO.parse(fasta_file, "fasta"):
    protein_ids.append(record.id)
    protein_sequences.append(str(record.seq))

print(f"共读取到 {len(protein_sequences)} 条蛋白质序列。")

# 4. 批量提取特征向量 (5120维)
# 虽然有 80GB 显存，但 15B 模型的计算开销极大，建议先从 batch_size=2 开始
batch_size = 2
all_embeddings = []

with torch.no_grad():
    for i in range(0, len(protein_sequences), batch_size):
        batch_seqs = protein_sequences[i:i+batch_size]

        # 词元化，超过模型最大支持长度会被截断
        inputs = tokenizer(batch_seqs, return_tensors="pt", padding=True, truncation=True)
        inputs = {k: v.to(model.device) for k, v in inputs.items()}

        outputs = model(**inputs)
        last_hidden_states = outputs.last_hidden_state

        # 均值池化压缩为全局特征
        attention_mask = inputs['attention_mask'].unsqueeze(-1).expand(last_hidden_states.size()).float()
        sum_embeddings = torch.sum(last_hidden_states * attention_mask, 1)
        sum_mask = attention_mask.sum(1)
        sum_mask = torch.clamp(sum_mask, min=1e-9)
        mean_pooled = sum_embeddings / sum_mask

        # 将结果转回 float32 并存入 CPU 内存
        all_embeddings.extend(mean_pooled.to(torch.float32).cpu().numpy())

        if (i + batch_size) % 50 == 0 or (i + batch_size) >= len(protein_sequences):
            print(f"已处理 {min(i + batch_size, len(protein_sequences))} / {len(protein_sequences)} 条序列...")

# 5. 保存为高维特征矩阵文件
embeddings_matrix = np.array(all_embeddings)
np.save("nocardioides_esm2_15B_embeddings.npy", embeddings_matrix)

print(f"提取完成！特征矩阵维度: {embeddings_matrix.shape} (预期应为 5252 x 5120)")

正在使用的计算设备: NVIDIA A100-SXM4-80GB
显存总量: 85.09 GB


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/725 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/95.0 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/93.0 [00:00<?, ?B/s]

...

Fetching 7 files:   0%|          | 0/7 [00:00<?, ?it/s]

model.safetensors.index.json: 0.00B [00:00, ?B/s]

Loading weights:   0%|          | 0/821 [00:00<?, ?it/s]

EsmModel LOAD REPORT from: facebook/esm2_t48_15B_UR50D
Key                         | Status     | 
----------------------------+------------+-
lm_head.layer_norm.weight   | UNEXPECTED | 
lm_head.dense.bias          | UNEXPECTED | 
lm_head.bias                | UNEXPECTED | 
lm_head.dense.weight        | UNEXPECTED | 
esm.embeddings.position_ids | UNEXPECTED | 
lm_head.layer_norm.bias     | UNEXPECTED | 
lm_head.decoder.weight      | UNEXPECTED | 
pooler.dense.weight         | MISSING    | 
pooler.dense.bias           | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


共读取到 5252 条蛋白质序列。
已处理 50 / 5252 条序列...
已处理 100 / 5252 条序列...
已处理 150 / 5252 条序列...
已处理 200 / 5252 条序列...
已处理 250 / 5252 条序列...
已处理 300 / 5252 条序列...
已处理 350 / 5252 条序列...
已处理 400 / 5252 条序列...
已处理 450 / 5252 条序列...
已处理 500 / 5252 条序列...
已处理 550 / 5252 条序列...
已处理 600 / 5252 条序列...
已处理 650 / 5252 条序列...
已处理 700 / 5252 条序列...
已处理 750 / 5252 条序列...
已处理 800 / 5252 条序列...
已处理 850 / 5252 条序列...
已处理 900 / 5252 条序列...
已处理 950 / 5252 条序列...
已处理 1000 / 5252 条序列...
已处理 1050 / 5252 条序列...
已处理 1100 / 5252 条序列...
已处理 1150 / 5252 条序列...
已处理 1200 / 5252 条序列...
已处理 1250 / 5252 条序列...
已处理 1300 / 5252 条序列...
已处理 1350 / 5252 条序列...
已处理 1400 / 5252 条序列...
已处理 1450 / 5252 条序列...
已处理 1500 / 5252 条序列...
已处理 1550 / 5252 条序列...
已处理 1600 / 5252 条序列...
已处理 1650 / 5252 条序列...
已处理 1700 / 5252 条序列...
已处理 1750 / 5252 条序列...
已处理 1800 / 5252 条序列...
已处理 1850 / 5252 条序列...
已处理 1900 / 5252 条序列...
已处理 1950 / 5252 条序列...
已处理 2000 / 5252 条序列...
已处理 2050 / 5252 条序列...
已处理 2100 / 5252 条序列...
已处理 2150 / 5252 条序列...
已处理 2200 / 52